# Classifier Evaluation: Keyword Baseline vs TF-IDF + Logistic Regression

Goals:
1. Verify answer-guided labels are correct
2. Train ML classifier and compare against keyword baseline
3. Inspect misclassified examples
4. Predict problem types for the test set

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

from src.data_utils import load_train, load_test, classify_problem
from src.classifier import PromptClassifier, evaluate_vs_keyword

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Answer-guided label distribution

In [ ]:
train = load_train(use_answer_labels=True)
print('Shape:', train.shape)
print(train['problem_type'].value_counts())

## 2. Train ML classifier (5-fold CV)

In [ ]:
clf = PromptClassifier()
clf.fit(train, verbose=True)

## 3. Keyword baseline vs ML — per-type accuracy

In [ ]:
results = evaluate_vs_keyword(train)
print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(results))
ax.bar([i - 0.2 for i in x], results['keyword_acc'], width=0.4, label='Keyword baseline')
ax.bar([i + 0.2 for i in x], results['ml_acc'], width=0.4, label='TF-IDF + LR')
ax.set_xticks(list(x))
ax.set_xticklabels(results['type'], rotation=20, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Classifier comparison per problem type')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Confusion matrix (ML classifier on train)

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

y_true = train['problem_type']
y_pred = clf.predict_batch(train['prompt'])
labels = sorted(y_true.unique())

cm = confusion_matrix(y_true, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=labels)
disp.plot(ax=ax, colorbar=False, xticks_rotation=30)
ax.set_title('Confusion matrix — ML classifier (train set)')
plt.tight_layout()
plt.show()

## 5. Inspect misclassifications

In [ ]:
train['ml_pred'] = clf.predict_batch(train['prompt'])
errors = train[train['ml_pred'] != train['problem_type']]
print(f'Misclassified: {len(errors)} / {len(train)} ({len(errors)/len(train):.2%})')

# Show a few errors per type
for gold, grp in errors.groupby('problem_type'):
    print(f'\n=== Gold: {gold} | Predicted as: {grp["ml_pred"].value_counts().to_dict()} ===')
    for _, row in grp.head(2).iterrows():
        print(f'  answer={row["answer"]!r}')
        print(f'  prompt={row["prompt"][:120]!r}')

## 6. Predict test set

In [ ]:
test = load_test()
test['ml_pred'] = clf.predict_batch(test['prompt'])

for _, row in test.iterrows():
    proba = clf.predict_proba(row['prompt'])
    top = sorted(proba.items(), key=lambda x: -x[1])
    print(f"id={row['id']} → {top[0][0]} ({top[0][1]:.2%}) | 2nd: {top[1][0]} ({top[1][1]:.2%})")
    print(f"  prompt: {row['prompt'][:100]!r}")